![Databricks Academy](./Includes/images/common/db-academy.png)

# 6 - Captura de Mudança de Dados com AUTO CDC e Dimensões de Mudança Lenta (SCD) TIPO 1

Nesta demonstração, continuaremos a construir nosso pipeline ingerindo dados de **clientes**. Os dados de clientes incluem novos clientes, clientes que excluíram suas contas e clientes que atualizaram suas informações (como endereço, e-mail, etc.). Precisaremos construir nosso pipeline de clientes implementando captura de mudança de dados (CDC) para dados de clientes usando SCD Tipo 1 (Tipo 2 está fora do escopo deste curso).

Fluxo do pipeline de clientes:

- A tabela bronze usa **Auto Loader** para ingerir dados JSON do armazenamento de objetos em nuvem com SQL (`FROM STREAM`).
- Uma tabela é definida para impor restrições antes de passar registros para a camada prata.
- `AUTO CDC` é usado para processar automaticamente dados CDC na camada prata como Tipo 1.
- Uma tabela ouro é definida para criar uma visão materializada dos clientes atuais com informações atualizadas (clientes excluídos, novos clientes e informações de clientes atualizadas).



### Objetivos de Aprendizagem

Ao final desta lição, você será capaz de:
- Aplicar a operação `AUTO CDC` em Lakeflow Spark Declarative Pipelines para processar captura de mudança de dados (CDC) integrando e atualizando dados recebidos de um fluxo de origem em uma tabela Delta existente, garantindo precisão e consistência dos dados.
- Analisar tabelas de Dimensões de Mudança Lenta (SCD Tipo 1) em Lakeflow Spark Declarative Pipelines para atualizar, inserir e excluir clientes em dados dimensionais, gerenciando o estado dos registros ao longo do tempo usando chaves apropriadas, versionamento e registros de tempo.


<div style="
  border-left: 4px solid #7b1fa2;
  background: #f3e5f5;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#4a148c; margin-bottom:6px; font-size: 1.1em;">Syntax Update</strong>
  <div style="color:#333;">

The AUTO CDC APIs replace the APPLY CHANGES APIs, and have the same syntax. The APPLY CHANGES APIs are still available, but Databricks recommends using the AUTO CDC APIs in their place.

The AUTO CDC APIs - Simplify change data capture with pipelines:
[AWS](https://docs.databricks.com/aws/en/ldp/cdc) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/cdc) |
[GCP](https://docs.databricks.com/gcp/en/ldp/cdc)


  </div>
</div>



## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

1. Run the following cell to configure your working environment for this course.

    This cell will also reset your `/Volumes/labuser/sdp_1_bronze/source` volume with the JSON files to the starting point, with one JSON file in each directory.

In [0]:
%run ./Includes/Classroom-Setup-REQUIRED

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-dd0e957f-35fd-428c-b268-e564686de09d
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-dd0e957f-35fd-428c-b268-e564686de09d
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,

✅ Vocareum workspace detected.
✅ Using existing Vocareum catalog: 'labuser15140516_1778971530'.



  STEP 1: Verifying catalog exists: labuser15140516_1778971530
  Catalog 'labuser15140516_1778971530' exists.

  STEP 2: Setting up 3 schema(s) in catalog: labuser15140516_1778971530
  [1/3] Checking: `labuser15140516_1778971530`.`sdp_1_bronze`... ALREADY EXISTS
  [2/3] Checking: `labuser15140516_1778971530`.`sdp_2_silver`... ALREADY EXISTS
  [3/3] Checking: `labuser15140516_1778971530`.`sdp_3_gold`... ALREADY EXISTS

  COMPLETE: 0 schema(s) created, 3 already existed.



DataFrame[]


  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines
  Checking: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data... FOUND


  STEP 1: Validating volume folder path...
  Found: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers

  STEP 2: Scanning for files...
  Found 1 file(s) to delete.

  STEP 3: Deleting files from: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers
  [1/1] Deleting: 00.json... DELETED

  COMPLETE: Deleted 1 file(s) from /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/la

Information,Value
Your Catalog:,labuser15140516_1778971530
Bronze Schema:,sdp_1_bronze
Silver Schema:,sdp_2_silver
Gold Schema:,sdp_3_gold
Source Volume:,/Volumes/labuser15140516_1778971530/sdp_1_bronze/source


Compute,Status,Details
Serverless,✓ Match,Version 5


## B. Explore the Customer Data Source Files

1. Run the cell below to programmatically view the files in your `/Volumes/labuser/sdp_1_bronze/source/customers` volume. 

    Confirm you only see one **00.json** file for customers.

In [0]:
%python
spark.sql(f'LIST "{source_volume_path}/customers"').display()

path,name,size,modification_time
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers/00.json,00.json,193413,1779235328000


2. Run the query below to explore the customers **00.json** file located at `/Volumes/labuser/sdp_1_bronze/source/customers`. Note the following:

   a. The file contains **939 customers** (remember this number).

   b. It includes general customer information such as **email**, **name**, and **address**.

   c. The **timestamp** column specifies the logical order of customer events in the source data.

   d. The **operation** column indicates whether the entry is for a new customer, a deletion, or an update.
      - **NOTE:** Since this is the first JSON file, all rows will be considered new customers.


In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/customers/00.json',
  format => "JSON"
)
ORDER BY operation;

address,city,customer_id,email,name,operation,state,timestamp,zip_code,_rescued_data
795 Hudson Islands,Long Beach,23820,btapia@example.net,Madison Larsen,NEW,CA,1639392557,55274,null
1733 Morales Turnpike,Chicopee,23871,michael71@example.net,Kimberly Fisher,NEW,MA,1639920828,38230,null
1968 Michael Summit,New York,23372,bushkristina@example.com,Matthew Tran,NEW,NY,1635348401,60875,null
1660 Rodriguez Village Apt. 697,Moreno Valley,23437,hartmandaniel@example.com,Denise Baker,NEW,CA,1635917577,65006,null
32591 Wanda Orchard,New York,23882,karenbryant@example.org,Joyce Brown,NEW,NY,1640030535,50044,null
28361 Brian Cove Apt. 023,Hanahan,23852,ilawson@example.net,Loretta Austin,NEW,SC,1639626891,57842,null
927 Thomas Street Suite 819,Hillsboro,23341,vanessa44@example.org,Bradley Bradshaw,NEW,OH,1635054174,23815,null
3034 Cruz Neck,Jacksonville,23373,deckersheila@example.com,John Hernandez,NEW,FL,1635420658,29910,null
0843 Snyder Centers,North Charleston,23404,mooreamy@example.com,Michael Anthony,NEW,SC,1635660960,21467,null
195 Shirley Cliff,Harrisonburg,23438,amandapearson@example.org,George Gutierrez,NEW,VA,1635977128,57682,null



<div style="
  border-left: 4px solid #7b1fa2;
  background: #f3e5f5;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#4a148c; margin-bottom:6px; font-size: 1.1em;">Question</strong>
  <div style="color:#333;">

How can we ingest new raw data source files (JSON) with customer updates into our pipeline to update the **customers_silver** table when inserts, updates, or deletes occur, without maintaining historical records (SCD Type 1)?

  </div>
</div>



## C. Change Data Capture with AUTO CDC APIs in Lakeflow Spark Declarative Pipelines

1. Run the cell below to create your starter Spark Declarative Pipeline for this demonstration. The pipeline will set the following for you:
    - Your default catalog: **labuser**
    - Your configuration parameter: `source` = `/Volumes/labuser/sdp_1_bronze/source/customers`

    **NOTES:** 
    - The `create_declarative_pipeline` function is a custom function built for this course to create the sample pipeline using the Databricks REST API. This avoids manually creating the pipeline and referencing the pipeline assets.

    - If the pipeline already exists, an error will be returned. In that case, you'll need to delete the existing pipeline and rerun this cell.

In [0]:
%python
create_declarative_pipeline(
    pipeline_name=f'6 - Change Data Capture with AUTO CDC - {my_catalog}',
    root_path_folder_name="6 - Change Data Capture with AUTO CDC Project",
    catalog_name=my_catalog,
    schema_name='default',
    source_folder_names=['orders', 'status', 'customers'],
    configuration={'source': source_volume_path}
)


  STEP 1: Checking for existing pipeline...
  ✅ No existing pipeline named '6 - Change Data Capture with AUTO CDC - labuser15140516_1778971530' found.

  STEP 2: Building pipeline configuration...
  Pipeline Name:    6 - Change Data Capture with AUTO CDC - labuser15140516_1778971530
  Catalog:          labuser15140516_1778971530
  Schema:           default
  Root Path:        /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/6 - Change Data Capture with AUTO CDC Project
  Source Folders:    orders, status, customers
  Serverless:       True
  Photon:           True
  Channel:          CURRENT
  Continuous:       False
  Development Mode: True
  Configuration:
    source = /Volumes/labuser15140516_1778971530/sdp_1_bronze/source

  STEP 3: Creating pipeline via API...
  ✅ Pipeline '6 - Change Data Capture with AUTO CDC - labuser15140516_1778971

2. Complete the following steps to open the starter Spark Declarative Pipeline project for this demonstration:

   a. In the main navigation bar, right-click on **Jobs & Pipelines** and select **Open Link in New Tab**.

   b. In **Jobs & Pipelines** select your **6 - Change Data Capture with AUTO CDC - labuser** pipeline.
      - **REQUIRED:** At the top near your pipeline name, turn on **New pipeline monitoring**.

   c. In the **Pipeline details** pane on the far right, select **Open in Editor** (field to the right of **Source code**) to open the pipeline in the **Lakeflow Pipeline Editor**.

   d. In the new tab, you should see the following folders:
      - **explorations**
      - **orders**
      - **status**
      - **customers**
      - Plus the extra **python_excluded** folder that contains the Python version.

   e. Open the **customers** folder and select the **customers_pipeline.sql** file.
      - **NOTE:** The **status** and **orders** pipelines are the same as we saw in the previous demonstrations.

## D. Spark Declarative Pipeline CDC SCD Type 1 Pipeline Steps
Follow the steps below using the **customers_pipeline.sql** file in the Lakeflow Pipelines editor.

1. Run the cell below and confirm each source volume (for **orders**, **status** and **customers**) contains a single JSON file.

In [0]:
%python
spark.sql(f'LIST "{source_volume_path}/orders"').display()
spark.sql(f'LIST "{source_volume_path}/status"').display()
spark.sql(f'LIST "{source_volume_path}/customers"').display()

path,name,size,modification_time
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders/00.json,00.json,15313,1779235334000


path,name,size,modification_time
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/status/00.json,00.json,40658,1779235340000


path,name,size,modification_time
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers/00.json,00.json,193413,1779235328000


### D1. PLEASE COMPLETE FIRST: Click the 'Run Pipeline' button to execute the Pipeline
1. To save some time, let's run the entire pipeline for **status**, **orders** and **customers**. Each volume contains 1 file.

    While the pipeline is running explore the code in the **customers/customers_pipeline.sql** for the new customers flow.

##### While the pipeline is running continue through the steps below to review the customer pipeline code.

### D2. ETAPA 1: JSON -> Ingestão Bronze (`customers_pipeline.sql`)
O código na **ETAPA 1** do arquivo **customers_pipeline.sql**:

   - Definimos uma tabela bronze de streaming chamada **customers_bronze_raw_demo6** usando uma fonte de dados configurada com Auto Loader (`FROM STREAM`) para ingerir incrementalmente arquivos do armazenamento em nuvem.

   - Adiciona a propriedade da tabela `pipelines.reset.allowed = false` para evitar a exclusão de todos os dados bronze ingeridos caso um refresh completo seja acionado.
   
   - Cria colunas para capturar o momento da ingestão dos dados e o nome do arquivo de origem para cada linha.


### D3. ETAPA 2: Crie a Tabela Bronze Clean Streaming com Regras de Qualidade de Dados

##### **NOTA:** Este exemplo mostra como usar técnicas avançadas de qualidade de dados com expectativas. Expectativas avançadas estão fora do escopo deste curso.

##### O código na **ETAPA 2** do arquivo **customers_pipeline.sql**:

- Adiciona três ações de violação de restrição: **WARN** (Aviso), **DROP** (Descartar), e **FAIL** (Falhar). Cada uma define como lidar com violações de restrição.
- Aplica múltiplas condições a uma única restrição.
- Usa uma função SQL embutida dentro de uma restrição.

##### Sobre a fonte de dados:

- Os dados são um feed CDC que contém operações de **`INSERT`**, **`UPDATE`** e **`DELETE`** para clientes.

  - REQUISITO: Operações **UPDATE** e **INSERT** devem conter entradas válidas para todos os campos.

  - REQUISITO: Operações **DELETE** devem conter valores **`NULL`** para todos os campos, exceto os campos **timestamp**, **customer_id** e **operation**.

  - Quando um registro vai ser descartado, todos os valores exceto o **customer_id** serão `null`.

    <div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
      <table style="width: 100%; border-collapse: collapse; font-size: 14pt; line-height: 1.5;">
        <thead>
          <tr style="background: #1B5162; color: white;">
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">endereço</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">cidade</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">customer_id</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">email</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">nome</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">operação</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">estado</th>
          </tr>
        </thead>
        <tbody>
          <tr style="background: #F9F7F4;">
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700; color: #0b2026;">23617</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">
              <span style="background: #FABFBA; color: #801C17; font-weight: 700; padding: 3px 10px; border-radius: 4px; font-size: 13pt;">DELETE</span>
            </td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
          </tr>
        </tbody>
      </table>

    </div>

<br></br>
**NOTA:** Para garantir que apenas dados válidos cheguem à nossa tabela silver, escreveremos uma série de regras de qualidade que permitem valores nulos esperados em operações **DELETE** enquanto rejeitam dados inválidos nos demais casos.



#### Vamos detalhar cada uma dessas restrições abaixo:


<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style>
.f1-cards { display: flex; gap: 8px; margin-bottom: 18px; flex-wrap: wrap; }
.f1-card {
  flex: 1; background: #F9F7F4; border-top: 6px solid transparent;
  border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent;
  border-radius: 8px; padding: 12px 10px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, background 0.15s; min-width: 140px;
}
.f1-card:hover { transform: translateY(-2px); }
.f1-card.active { background: #fff; border-left-color: var(--dc); border-right-color: var(--dc); border-bottom-color: var(--dc); }
.f1-card-label { display: block; font-size: 13pt; font-weight: 700; color: #0b2026; line-height: 1.3; pointer-events: none; }
.f1-card-sub { display: block; font-size: 11.5pt; font-weight: 400; color: #618794; margin-top: 4px; pointer-events: none; }
.f1-layout { display: flex; gap: 22px; align-items: stretch; }
.f1-code-wrap { flex: 1; position: relative; }
.f1-theme-btn {
  position: absolute; top: 10px; right: 12px; z-index: 2;
  background: rgba(255,255,255,0.12); border: 1px solid rgba(255,255,255,0.2);
  border-radius: 6px; padding: 5px 12px; font-size: 11pt; font-weight: 600; color: #cdd6f4;
  cursor: pointer; transition: background 0.15s, color 0.15s, border-color 0.15s;
}
.f1-theme-btn:hover { background: rgba(255,255,255,0.2); }
.f1-code-wrap.light .f1-theme-btn { background: rgba(0,0,0,0.06); border-color: rgba(0,0,0,0.15); color: #444; }
.f1-code-wrap.light .f1-theme-btn:hover { background: rgba(0,0,0,0.1); }
.f1-code {
  border-radius: 10px; padding: 20px 22px; font-family: 'Menlo','Consolas',monospace;
  font-size: 12pt; line-height: 1.75; overflow-x: auto; background: #1e1e2e; color: #cdd6f4;
  transition: background 0.3s, color 0.3s;
}
.f1-code-wrap.light .f1-code { background: #fafafa; color: #383a42; }
.f1-code .tk-kw { color: #cba6f7; }
.f1-code .tk-fn { color: #89b4fa; }
.f1-code .tk-str { color: #a6e3a1; }
.f1-code .tk-num { color: #fab387; }
.f1-code .tk-cmt { color: #6c7086; }
.f1-code .tk-dim { color: #a6adc8; }
.f1-code-wrap.light .f1-code .tk-kw { color: #a626a4; }
.f1-code-wrap.light .f1-code .tk-fn { color: #4078f2; }
.f1-code-wrap.light .f1-code .tk-str { color: #50a14f; }
.f1-code-wrap.light .f1-code .tk-num { color: #986801; }
.f1-code-wrap.light .f1-code .tk-cmt { color: #a0a1a7; }
.f1-code-wrap.light .f1-code .tk-dim { color: #696c77; }
.f1-code .line { display: block; padding: 1px 6px; border-radius: 3px; transition: background 0.25s, opacity 0.25s; }
.f1-code.has-highlight .line { opacity: 0.25; }
.f1-code.has-highlight .line.hl { opacity: 1; background: rgba(255,255,255,0.08); }
.f1-code-wrap.light .f1-code.has-highlight .line.hl { background: rgba(0,0,0,0.06); }
.f1-explain { flex: 0 0 320px; display: flex; flex-direction: column; justify-content: center; }
.f1-explain-card { background: #F9F7F4; border-radius: 10px; border-top: 6px solid #ccc; padding: 20px; font-size: 14pt; line-height: 1.6; opacity: 0; transition: opacity 0.3s; min-height: 200px; }
.f1-explain-card.visible { opacity: 1; }
.f1-explain-card ul { margin: 10px 0 0 0; padding-left: 20px; }
.f1-explain-card li { margin-bottom: 10px; }
.f1-badge {
  display: inline-block; padding: 3px 10px; border-radius: 999px;
  font-size: 11.5pt; font-weight: 700; margin-bottom: 12px;
}
</style>

<!-- Header -->
<div style="background: #1B5162; color: white; border-radius: 8px 8px 4px 4px; padding: 20px 24px; margin-bottom: 18px;">
  <div style="font-size: 20pt; font-weight: 700;">Restrições de Qualidade de Dados - Tabela Bronze Clean</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;">Cada restrição define uma regra e o que acontece quando um registro a viola. Clique em uma restrição para explorar.</div>
</div>

<div class="f1-cards">
  <div class="f1-card" data-id="0" onclick="bcSelect(0)" style="--dc:#98102A; border-top-color:#98102A;">
    <span class="f1-card-label"><code>valid_id</code></span>
    <span class="f1-card-sub">FAIL UPDATE</span>
  </div><div class="f1-card" data-id="1" onclick="bcSelect(1)" style="--dc:#FF5F46; border-top-color:#FF5F46;">
    <span class="f1-card-label"><code>valid_operation</code></span>
    <span class="f1-card-sub">DROP ROW</span>
  </div><div class="f1-card" data-id="2" onclick="bcSelect(2)" style="--dc:#FFAB00; border-top-color:#FFAB00;">
    <span class="f1-card-label"><code>valid_name</code></span>
    <span class="f1-card-sub">WARN (padrão)</span>
  </div><div class="f1-card" data-id="3" onclick="bcSelect(3)" style="--dc:#4299E0; border-top-color:#4299E0;">
    <span class="f1-card-label"><code>valid_address</code></span>
    <span class="f1-card-sub">WARN (padrão)</span>
  </div><div class="f1-card" data-id="4" onclick="bcSelect(4)" style="--dc:#00A972; border-top-color:#00A972;">
    <span class="f1-card-label"><code>valid_email</code></span>
    <span class="f1-card-sub">DROP ROW</span>
  </div>
</div>

<div class="f1-layout">
  <div class="f1-code-wrap" id="bc-code-wrap">
    <button class="f1-theme-btn" id="bc-theme-btn" onclick="bcToggle()">Modo Claro</button>
    <div class="f1-code" id="bc-code">
      <span class="line" data-g="all"><span class="tk-kw">CREATE STREAMING TABLE</span> 1_bronze_db.customers_bronze_clean_demo6</span>
      <span class="line" data-g="all">&nbsp;&nbsp;(</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_id</span> <span class="tk-kw">EXPECT</span> (customer_id <span class="tk-kw">IS NOT NULL</span>) <span class="tk-kw">ON VIOLATION FAIL UPDATE</span>,</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_operation</span> <span class="tk-kw">EXPECT</span> (operation <span class="tk-kw">IS NOT NULL</span>) <span class="tk-kw">ON VIOLATION DROP ROW</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_name</span> <span class="tk-kw">EXPECT</span> (name <span class="tk-kw">IS NOT NULL OR</span> operation = <span class="tk-str">"DELETE"</span>),</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_address</span> <span class="tk-kw">EXPECT</span> (</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(address <span class="tk-kw">IS NOT NULL AND</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;city <span class="tk-kw">IS NOT NULL AND</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;state <span class="tk-kw">IS NOT NULL AND</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;zip_code <span class="tk-kw">IS NOT NULL</span>) <span class="tk-kw">OR</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;operation = <span class="tk-str">"DELETE"</span>),</span>
      <span class="line" data-g="4">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_email</span> <span class="tk-kw">EXPECT</span> (</span>
      <span class="line" data-g="4">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;rlike(email, <span class="tk-str">'^([a-zA-Z0-9_\\-\\.]+)@([a-zA-Z0-9_\\-\\.]+)\\.([a-zA-Z]{2,5})$'</span>) <span class="tk-kw">OR</span></span>
      <span class="line" data-g="4">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;operation = <span class="tk-str">"DELETE"</span>) <span class="tk-kw">ON VIOLATION DROP ROW</span></span>
      <span class="line" data-g="all">&nbsp;&nbsp;)</span>
      <span class="line" data-g="all">&nbsp;&nbsp;<span class="tk-kw">COMMENT</span> <span class="tk-str">"Limpa a coluna de timestamp do bronze bruto e adiciona restrições de qualidade de dados"</span></span>
      <span class="line" data-g="all"><span class="tk-kw">AS</span></span>
      <span class="line" data-g="all"><span class="tk-kw">SELECT</span></span>
      <span class="line" data-g="all">&nbsp;&nbsp;*,</span>
      <span class="line" data-g="all">&nbsp;&nbsp;<span class="tk-fn">CAST</span>(from_unixtime(timestamp) <span class="tk-kw">AS</span> timestamp) <span class="tk-kw">AS</span> timestamp_datetime</span>
      <span class="line" data-g="all"><span class="tk-kw">FROM STREAM</span> 1_bronze_db.customers_bronze_raw_demo6;</span>
    </div>
  </div>
  <div class="f1-explain">
    <div class="f1-explain-card" id="bc-explain-card"></div>
  </div>
</div>

</div>

<script>
var BC_DATA = [
  {
    color: '#98102A',
    badge: 'FAIL UPDATE',
    badgeBg: 'rgba(152,16,42,0.12)',
    title: 'valid_id',
    text: 'A ação de violação mais rígida. Se algum registro chegar com <strong>null <code>customer_id</code></strong>, toda a atualização do micro-lote falha.',
    bullets: [
      'Nenhum registro desse lote é gravado',
      'Use para campos realmente indispensáveis, como chave primária',
      'Prefira quando dados ruins devem interromper o pipeline, não passar silenciosamente'
    ]
  },
  {
    color: '#FF5F46',
    badge: 'DROP ROW',
    badgeBg: 'rgba(255,95,70,0.12)',
    title: 'valid_operation',
    text:       'Remove silenciosamente qualquer registro onde <code>operation</code> é nulo. O pipeline continua rodando e apenas a linha problemática é descartada.',
    bullets: [
      'Contagens de violações são rastreadas nas métricas do pipeline',
      'Use quando linhas ruins devem ser excluídas mas não interromper o processamento',
      'Uma <code>operation</code> nula significa que não podemos aplicar a lógica CDC — então não há uma forma segura de lidar'
    ]
  },
  {
    color: '#FFAB00',
    badge: 'WARN (padrão)',
    badgeBg: 'rgba(255,171,0,0.12)',
    title: 'valid_name',
    text: 'Marca registros onde <code>name</code> é nulo e a operação não é DELETE. Nenhuma linha é descartada — violações são rastreadas apenas nas métricas.',
    bullets: [
      'Sem cláusula <code>ON VIOLATION</code> = comportamento WARN por padrão',
      'O <code>OR operation = "DELETE"</code> permite nulos esperados para deletes',
      'Use quando quiser visibilidade sobre problemas de qualidade sem interromper o pipeline'
    ]
  },
  {
    color: '#4299E0',
    badge: 'WARN (padrão)',
    badgeBg: 'rgba(66,153,224,0.12)',
    title: 'valid_address',
    text: 'Verifica todos os quatro campos de endereço de uma vez. Um registro passa se todos forem não nulos, OU se a operação for DELETE (onde nulos são esperados).',
    bullets: [
      'Múltiplas condições combinadas com AND em uma única restrição',
      'O OR permite que registros DELETE ignorem a validação de endereço',
      'Violações são registradas nas métricas mas linhas não são descartadas'
    ]
  },
  {
    color: '#00A972',
    badge: 'DROP ROW',
    badgeBg: 'rgba(0,169,114,0.12)',
    title: 'valid_email',
    text: 'Usa <code>rlike()</code>, uma função SQL regex para validar o formato do email. Registros com email inválido são descartados.',
    bullets: [
      'O padrão regex corresponde a formatos padrão de email',
      'Operações DELETE são isentas (o campo email será nulo)',
      'Linhas descartadas terão todos os campos nulos exceto <code>customer_id</code>'
    ]
  }
];

var bcCurrent = null, bcLight = false;

function bcToggle() {
  bcLight = !bcLight;
  document.getElementById('bc-code-wrap').classList.toggle('light', bcLight);
  document.getElementById('bc-theme-btn').textContent = bcLight ? 'Modo Escuro' : 'Modo Claro';
}

function bcSelect(id) {
  var code = document.getElementById('bc-code');
  var card = document.getElementById('bc-explain-card');
  document.querySelectorAll('.f1-card').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id);
  });
  if (bcCurrent === id) {
    code.classList.remove('has-highlight');
    card.classList.remove('visible');
    document.querySelectorAll('.f1-card').forEach(function(b) { b.classList.remove('active'); });
    bcCurrent = null;
    return;
  }
  bcCurrent = id;
  var c = BC_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln) {
    var g = ln.dataset.g;
    ln.classList.toggle('hl', g === String(id) || g === 'all');
  });
  var bulletsHtml = c.bullets.map(function(b) { return '<li>' + b + '</li>'; }).join('');
  card.style.borderTopColor = c.color;
  card.innerHTML =
    '<span class="f1-badge" style="background:' + c.badgeBg + ';color:' + c.color + ';">' + c.badge + '</span>' +
    '<div style="font-size:17pt;font-weight:700;margin-bottom:10px;color:#0b2026;font-family:monospace;">' + c.title + '</div>' +
    '<div style="margin-bottom:10px;">' + c.text + '</div>' +
    '<ul>' + bulletsHtml + '</ul>';
  card.classList.add('visible');
}
</script>

### D4. ETAPA 3: Processando Dados CDC com **`AUTO CDC INTO`**
Spark Declarative Pipelines introduz uma nova estrutura sintática para simplificar o processamento de feeds CDC: `AUTO CDC INTO` (anteriormente `APPLY CHANGES INTO`).

O código na **ETAPA 3** do arquivo **customers_pipeline.sql** usa `AUTO CDC INTO` para:

- **CRIA**
  - A tabela streaming **sdp_2_silver.scd_type_1_customers_silver_demo6** se ela não existir,

- **ATUALIZA**
  - A tabela streaming **sdp_2_silver.scd_type_1_customers_silver_demo6** com atualizações, inserções e deleções usando registros da tabela streaming **sdp_1_bronze.customers_bronze_clean_demo6**.

#### Notas Adicionais
**`AUTO CDC INTO`** possui as seguintes garantias e requisitos:
- Realiza ingestão incremental/streaming de dados CDC
- Oferece sintaxe simples para especificar um ou mais campos como chave primária de uma tabela
- Assume por padrão que as linhas conterão inserções e atualizações
- Pode aplicar deleções opcionalmente
- Ordena automaticamente registros que chegam atrasados usando uma chave de ordenação fornecida pelo usuário (ordem de processamento das linhas)
- Usa sintaxe simples para especificar colunas a serem ignoradas com a palavra-chave **`EXCEPT`**
- O padrão para aplicação de mudanças é SCD Tipo 1. Você também pode usar SCD Tipo 2 se desejar. Aqui focaremos no SCD Tipo 1.

#### Documentação
AUTO CDC INTO (Lakeflow Spark Declarative Pipelines):
[AWS](https://docs.databricks.com/aws/en/dlt-ref/dlt-sql-ref-apply-changes-into) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/developer/ldp-sql-ref-apply-changes-into) |
[GCP](https://docs.databricks.com/gcp/en/dlt-ref/dlt-sql-ref-apply-changes-into)

APIs AUTO CDC - Simplificam captura de dados de mudança com Lakeflow Spark Declarative Pipelines:
[AWS](https://docs.databricks.com/aws/en/dlt/cdc) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/cdc) |
[GCP](https://docs.databricks.com/gcp/en/dlt/cdc)

### D5. STEP 4: Explore the Customers Pipeline Graph
After running the pipeline and reviewing the code cells, take time to explore the pipeline results for the **customers** flow following the steps below.

**Run with 1 JSON File**

![demo6_cdc_run01.png](./Includes/images/change-data-capture/demo6_cdc_run_1.png)



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    View the Results
  </strong>
  <div style="color:#333;">

Notice the following:
1. In the **customers** flow in the pipeline graph, notice that **939** rows were streamed into the three streaming tables.
    - This is because all records are new and valid entries, they were ingested throughout the flow.

2. In the table window below, find the **scd_type_1_customers_silver_demo6** table and select **Table metrics**. 

    Note the following:

    - The **Upserted** column indicates that all **939** rows were upserted into the table, as all rows are new.
  </div>
</div>


### D6. STEP 5: Explore the Customers Pipeline Tables

1. Run the query below to view the **scd_type_1_customers_silver_demo6** streaming table (the table with SCD Type 1 updates, inserts and deletes).

    Notice the following after the first run ingestion the **00.json** file:

   - The streaming table contains all **939 rows** from the **00.json** file, since they are all new customers being added to the target table.

   - Each record was inserted into the empty streaming table.

In [0]:
SELECT *
FROM sdp_2_silver.scd_type_1_customers_silver_demo6;

address,city,customer_id,email,name,state,zip_code,processing_time,source_file,timestamp_datetime
241 Dennis Springs,Springfield,22122,marie21@example.net,Cynthia Price,MA,69026,2026-05-20T00:09:02.767Z,00.json,2021-12-31T20:25:14.000Z
80972 Johnson Island,Denver,22141,michelle30@example.net,Stephen Parker,CO,76478,2026-05-20T00:09:02.767Z,00.json,2021-12-29T10:48:27.000Z
3772 Miller Junctions Apt. 383,McAllen,22144,phillipsolivia@example.com,Angela Frost,TX,57649,2026-05-20T00:09:02.767Z,00.json,2021-12-29T10:17:00.000Z
8554 Summer Plain Suite 213,Delano,22161,johnstonkatherine@example.net,Brittany Schneider,CA,74021,2026-05-20T00:09:02.767Z,00.json,2021-12-27T07:44:04.000Z
399 Jackson Villages,Santa Fe Springs,22163,ronaldfrazier@example.org,Jason Anderson,CA,38777,2026-05-20T00:09:02.767Z,00.json,2021-12-27T22:56:52.000Z
610 Thompson Valleys,Las Vegas,22169,fisherdebra@example.net,Steven Mullen,NV,74024,2026-05-20T00:09:02.767Z,00.json,2021-12-27T14:56:48.000Z
471 Mcdonald Corner Apt. 354,Winchester,22195,jason50@example.com,David West,VA,80381,2026-05-20T00:09:02.767Z,00.json,2021-12-24T08:49:56.000Z
845 Ralph Garden,Falfurrias,22196,sydney25@example.org,Daniel Berger,TX,15928,2026-05-20T00:09:02.767Z,00.json,2021-12-24T20:30:57.000Z
400 Bryant Mountain,New York,22198,christina16@example.net,David Lee,NY,41010,2026-05-20T00:09:02.767Z,00.json,2021-12-23T20:10:42.000Z
137 Nicholas Vista Apt. 083,Port Huron,22203,shepardseth@example.net,Shane Hensley,MI,41717,2026-05-20T00:09:02.767Z,00.json,2021-12-23T11:45:43.000Z


2. Query the **scd_type_1_customers_silver_demo6** streaming table for the following **customer_id** values (*23225*, *23617*).

   Notice the following:
      - **customer_id** = *23225*
         - **Address**: `76814 Jacqueline Mountains Suite 815`
         - **State**: `TX`
      - **customer_id** = *23617*
         - This customer exists in the first execution (in file **00.json**)

In [0]:
SELECT *
FROM sdp_2_silver.scd_type_1_customers_silver_demo6
WHERE customer_id IN (23225, 23617);

address,city,customer_id,email,name,state,zip_code,processing_time,source_file,timestamp_datetime
76814 Jacqueline Mountains Suite 815,El Paso,23225,andrewcarter@example.org,Sandy Adams,TX,46521,2026-05-20T00:09:02.767Z,00.json,2021-10-12T10:52:25.000Z
0727 Michael Locks,Detroit,23617,juan69@example.com,Stephen Green,MI,25064,2026-05-20T00:09:02.767Z,00.json,2021-11-22T17:06:38.000Z


## E. Land New Data to Your Data Source Volume
Complete the following after executing and reviewing the **customers** pipeline flow that consisted of ingesting one file (**00.json**) from cloud storage.

1. Run the cell below to land a new JSON file to each volume (**customers**, **status** and **orders**) to simulate new files being added to your cloud storage locations.

In [0]:
%python

## Find data in workspace data folder
data_path = find_folder('Includes/data')

## Land JSON files to your orders volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/orders',
    target_volume_path=f'{source_volume_path}/orders',
    n=2
)

## Land JSON files to your status volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/status',
    target_volume_path=f'{source_volume_path}/status',
    n=2
)

## Land JSON files to your customers volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/customers',
    target_volume_path=f'{source_volume_path}/customers',
    n=2
)


  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines
  Checking: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data... FOUND


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data/orders

  STEP 2: Checking target volume path...
  Target volume path already exists: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders

  STEP 3: Reading source files...
  Found 5 file(s) in source folder.

  STEP 4

2. Run the cell below to programmatically view the files in your `/Volumes/labuser/sdp_1_bronze/source/customers` volume.

    Confirm your volume now contains the original **00.json** file and the new **01.json** file.

In [0]:
%python
spark.sql(f'LIST "{source_volume_path}/customers"').display()

path,name,size,modification_time
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers/00.json,00.json,193413,1779235328000
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers/01.json,01.json,4709,1779236089000


3. Run the cell to explore the raw data in the new **01.json** file prior to ingesting it in your pipeline.

   Notice the following:

   - This file contains **23** rows.

   - The **operation** column specifies **UPDATE**, **DELETE**, and **NEW** operations for customers.
      - **In the new 01.json file there are**:
         - 12 customers with **UPDATE** values
         - 1 customer with a **DELETE** value
         - 10 new customers with a **NEW** value

In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/customers/01.json',
  format => "JSON"
)
ORDER BY customer_id;

address,city,customer_id,email,name,operation,state,timestamp,zip_code,_rescued_data
618 Villarreal Stravenue Suite 601,Pittsburgh,22668,medinaryan@example.org,Steven Conway,UPDATE,PA,1641011454,28488,null
451 Hunt Station,Johnson City,22760,ashley44@example.org,Michael Lewis,UPDATE,TN,1641057420,90476,null
80805 Mcmillan Street,Maryville,22931,flowersjose@example.org,Teresa Mooney,UPDATE,TN,1641043886,77378,null
512 John Stravenue Suite 239,Kingsport,23225,andrewcarter@example.org,Sandy Adams,UPDATE,TN,1641067642,94660,null
510 Martin Gardens Apt. 723,Fullerton,23345,wnunez@example.net,Andrew Perez,UPDATE,CA,1641025081,30769,null
077 Linda Corners,Detroit,23439,mstone@example.net,Lori Jordan,UPDATE,MI,1641066875,20864,null
null,null,23617,null,null,DELETE,null,1641054281,null,null
50434 Turner Land Suite 696,New York,23666,faulknershannon@example.net,John Rodgers,UPDATE,NY,1641045935,66088,null
976 Lester Heights Suite 317,Austin,23768,conleycarly@example.org,Steven Campbell,UPDATE,TX,1641036961,08533,null
80101 Adam Spur Apt. 971,Chicago,23789,marcus91@example.org,Robert Smith,UPDATE,IL,1641039413,81100,null


4. Run the cell to view **customer_id** values *23225* and *23617* in the **01.json** file.

   - In the results below, find the row with **customer_id** *23225* and note the following:

      - The original address for **Sandy Adams** (from the streaming table, file **00.json**) was: `76814 Jacqueline Mountains Suite 815`, `TX`
      - The updated address for **Sandy Adams** (from the file below) is: `512 John Stravenue Suite 239`, `TN`

   - In the results below, find the row with **customer_id** *23617* and note the following:
      - The **operation** for this customer is **DELETE**.
      - When the **operation** column is delete, all other column values are `null`.


In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/customers/01.json',
  format => "JSON"
)
WHERE customer_id IN (23225, 23617)
ORDER BY customer_id;

address,city,customer_id,email,name,operation,state,timestamp,zip_code,_rescued_data
512 John Stravenue Suite 239,Kingsport,23225,andrewcarter@example.org,Sandy Adams,UPDATE,TN,1641067642,94660,null
null,null,23617,null,null,DELETE,null,1641054281,null,null


### E1. Run the SDP with the New File

##### Go back to your pipeline and click `Run pipeline` button to ingest the new JSON file (**01.json**) incrementally and perform CDC SCD Type 1 on the `scd_type_1_customers_silver_demo6` table.

## F. Explore the Customers Pipeline

After you have explored and landed 1 new JSON file into each of your cloud data sources, complete the following to explore the **customers** flow in the **Pipeline graph**:

a. 23 rows were read into the:

  - **customers_bronze_raw_demo6** streaming table
  - **customers_bronze_clean_demo6** streaming table (all data quality checks passed)
  - The pipeline only ingested and processed the NEW **01.json** file

b. In the **scd_type_1_customers_silver_demo6** streaming table details (The CDC SCD Type 1 table) it contains:
  - **Upserted = 22**:
    - 12 customers with UPDATE values (previous customer were simply updated with the new values)
    - 10 new customers with a NEW value (new customers were inserted into the table)
  - **Deleted records = 1**:
    - 1 customer was marked as DELETE and deleted from the table

![Run 2](./Includes/images/change-data-capture/demo6_cdc_run_2.png)

## G. Explore the CDC SCD Type 1 on the `scd_type_1_customers_silver_demo6` Streaming Table

1. View the data in the **scd_type_1_customers_silver_demo6** streaming table with SCD Type 1 and observe the following:

   a. The table contains **948 rows**:
      - **initial 939 customers**
      - \+ **10** new customers
      - \- **1** deleted customer
      - **NOTES:**
         - The **12** updates to original customers were made in place and updated the original record (SCD Type 1 does not keep historical records).
         - The **1** record marked for deletion was deleted from the table.

In [0]:
SELECT customer_id, address, name
FROM sdp_2_silver.scd_type_1_customers_silver_demo6;

2. Query the **sdp_2_silver.scd_type_1_customers_silver_demo6** table for the following **customer_id** values: *23225* and *23617*. These were the values we reviewed earlier.

    Notice the following:

    - **customer_id** *23225* has been updated to the new address. The historical address was not retained because we used SCD Type 1.
    - **customer_id** *23617* has been deleted from the table. It no longer exists because we used SCD Type 1.


In [0]:
SELECT *
FROM sdp_2_silver.scd_type_1_customers_silver_demo6
WHERE customer_id IN (23225, 23617);

address,city,customer_id,email,name,state,zip_code,processing_time,source_file,timestamp_datetime
512 John Stravenue Suite 239,Kingsport,23225,andrewcarter@example.org,Sandy Adams,TN,94660,2026-05-20T00:15:55.839Z,01.json,2022-01-01T20:07:22.000Z


## Additional Resources

- What is change data capture (CDC)?:
[AWS](https://docs.databricks.com/aws/en/dlt/what-is-change-data-capture) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/what-is-change-data-capture) |
[GCP](https://docs.databricks.com/gcp/en/dlt/what-is-change-data-capture)

- AUTO CDC INTO (Lakeflow Spark Declarative Pipelines) documentation:
[AWS](https://docs.databricks.com/aws/en/dlt-ref/dlt-sql-ref-apply-changes-into) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/developer/ldp-sql-ref-apply-changes-into) |
[GCP](https://docs.databricks.com/gcp/en/dlt-ref/dlt-sql-ref-apply-changes-into)

- The AUTO CDC APIs - Simplify change data capture with Lakeflow Spark Declarative Pipelines documentation:
[AWS](https://docs.databricks.com/aws/en/dlt/cdc) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/cdc) |
[GCP](https://docs.databricks.com/gcp/en/dlt/cdc)

- [How to implement Slowly Changing Dimensions when you have duplicates - Part 1: What to look out for?](https://community.databricks.com/t5/technical-blog/how-to-implement-slowly-changing-dimensions-when-you-have/ba-p/40568)


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>